# 6. Fit the height of a bunny's ear under load

Pull upward on the tip of the tetrahedral bunny's taller ear with a
prescribed force. Start with lesson 5's trained material and optimize
its 1,216 weights and biases until the highest point of the loaded bunny
reaches a specified height. Each trial $\theta$ gives a new equilibrium
$u^*(\theta)$ under the same load and clamp.

## Load lesson 5's trained weights

Use the `trained_theta.npy` exported by lesson 5. For the downloaded
lesson, keep the weights and `bunny-tet.veg` beside the notebook.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse as sp
import torch
from matplotlib.collections import PolyCollection
from scipy.optimize import brentq
from torch import nn
%matplotlib inline
from litepgo import dof, energy, fem, mesh, solver
from litepgo.torch import TorchVolumetricMaterial, response

torch.set_num_threads(1)
lesson_dir = Path('tutorials/torch') if Path('tutorials/torch').is_dir() else Path('.')
trained_theta = np.load(lesson_dir / 'trained_theta.npy', allow_pickle=False)

## Define the learned FEM material

Recreate lesson 5's invariant Softplus network with the same layer
order. Each material point receives its own row of `theta`. The network
value, reference value, and reference slope all use that row. Subtract the
reference value and the slope along $\log J$ so the energy and stress
vanish at $F_e=I$.

In [ ]:
def invariants(Fe):
    J = torch.linalg.det(Fe)
    C = Fe.transpose(-1, -2) @ Fe
    I1 = C.diagonal(dim1=-2, dim2=-1).sum(-1)
    I2 = 0.5 * (I1.square() - C.square().sum(dim=(-2, -1)))
    return torch.stack(
        (J.pow(-2 / 3) * I1 - 3, J.pow(-4 / 3) * I2 - 3, J.log()), dim=-1,
    )


def network_value(z, theta):
    w1, b1, w2, b2, w3 = theta.split((32 * 3, 32, 32 * 32, 32, 32))
    h1 = nn.functional.softplus(nn.functional.linear(z, w1.reshape(32, 3), b1))
    h2 = nn.functional.softplus(nn.functional.linear(h1, w2.reshape(32, 32), b2))
    return nn.functional.linear(h2, w3.reshape(1, 32)).squeeze(-1)


network_gradient = torch.func.grad(network_value, argnums=0)


class LearnedMaterial(TorchVolumetricMaterial):
    num_parameters = 1216

    @staticmethod
    @torch.vmap
    def W(F, Fp, theta):
        z = invariants(F @ torch.linalg.inv(Fp))
        origin = torch.zeros_like(z)
        reference_value = network_value(origin, theta)
        reference_slope = network_gradient(origin, theta)[2]
        elastic_energy = network_value(z, theta) - reference_value - reference_slope * z[2]
        return torch.linalg.det(Fp) * elastic_energy

num_parameters = LearnedMaterial.num_parameters
assert trained_theta.shape == (num_parameters,) and trained_theta.dtype == np.float64
assert np.isfinite(trained_theta).all()

## Clamp the bunny and pull on the taller ear

The mesh's original $y$ coordinate is vertical. Clamp vertices at or
below 92% of the rest height range. Tetrahedra with four clamped
vertices contribute zero to the free equilibrium equations, so keep
only the tetrahedra incident to a free vertex.

Distribute a fixed total upward force over the original surface vertices
above 97% of the rest height range. On this mesh they lie on the taller
ear. The external potential is $-f^\mathsf{T}u$.

In [ ]:
asset_dir = next(folder for folder in (Path('.'), Path('assets'), Path('../../assets'))
                 if (folder / 'bunny-tet.veg').is_file())
bunny = mesh.read_veg(asset_dir / 'bunny-tet.veg')
y = bunny.positions[:, 1]
height_fraction = (y - y.min()) / np.ptp(y)
clamped = height_fraction <= .92
cells = bunny.topology.cells
active_cells = cells[(~clamped[cells]).any(axis=1)]
vertices, local_cells = np.unique(active_cells, return_inverse=True)
X = bunny.positions[vertices]
topology = mesh.TetTopology(len(X), local_cells.reshape(-1, 4).astype(np.int32))
mapping = dof.VertexDofMap(topology, 3)
basis = fem.TetP1Basis
material = LearnedMaterial()
key = (basis, fem.TetCentroidQuadrature)
geometry = [fem.InterpolatedResponse[basis](mapping.gather(e, X.ravel()))
            for e in range(topology.num_cells)]
plastic = [energy.ConstantResponse(3, np.eye(3).ravel(order='F'))] * topology.num_cells

fixed_mask = np.repeat(clamped[vertices], 3)
fixed = dof.DofReduction(mapping.num_dofs, np.flatnonzero(fixed_mask).tolist())
extracted_surface = mesh.extract_surface_with_vertices(bunny)
surface = np.asarray(extracted_surface.vertices)
loaded = np.isin(vertices, surface) & (height_fraction[vertices] > .97)
assert loaded.any() and not clamped[vertices[loaded]].any()

total_force_y = .0004
load = np.zeros_like(X)
load[loaded, 1] = total_force_y / loaded.sum()
zero_quadratic = sp.csc_matrix((mapping.num_dofs, mapping.num_dofs))
external = energy.QuadraticEnergy(zero_quadratic, -load.ravel())
print(f'Pulling {loaded.sum()} surface vertices; {topology.num_cells} active tetrahedra')

## Build the equilibrium response

As in lesson 4, `make_energy(theta)` supplies the total potential for
Newton, and `make_response(u)` supplies the derivative of internal force
with respect to the same $\theta$ at fixed $u$. The prescribed load has
zero derivative with respect to the network weights.

In [ ]:
parameter_map = dof.ConstantDofMap(topology, num_parameters)

def make_energy(theta, load_energy=external):
    theta_field = energy.ConstantResponse(3, theta)
    elastic = fem.VolumetricFemEnergy[key](material, mapping, geometry, plastic, [theta_field] * topology.num_cells)
    return energy.WeightedSumEnergy([(elastic, 1.), (load_energy, 1.)])

def make_response(u):
    return fem.VolumetricFemForceMaterialResponse[key](material, parameter_map, mapping, geometry, u, plastic)

zero_displacement = np.zeros(mapping.num_dofs)
newton_options = solver.NewtonOptions(gradient_tolerance=1e-11, max_iterations=60)
equilibrium = solver.StaticEquilibriumResponse(
    num_parameters=num_parameters,
    fixed=fixed,
    initial=zero_displacement,
    options=newton_options,
    make_energy=make_energy,
    make_response=make_response,
)

## Match the maximum height at equilibrium

`target_height` is the desired absolute $y$ coordinate in the mesh's
units. Compute the actual maximum over the deformed vertices and the
clamped rest vertices. For linear tetrahedra, the highest point is
attained at a vertex. The loss is half the squared height error, scaled
by a fixed length of $10^{-3}$ in mesh units.

`response(equilibrium, theta)` solves for the displacement and applies
the native adjoint in backward. With a unique highest vertex, the
height gradient follows that vertex. Check the gradient with a
directional finite difference through two perturbed equilibrium solves.

In [ ]:
target_height = .200
height_scale = .001
rest_y = torch.from_numpy(X[:, 1].copy())
clamped_height = rest_y.new_tensor(y[clamped].max())
theta = torch.from_numpy(trained_theta.copy()).requires_grad_()

def max_height(u):
    return torch.maximum((rest_y + u[1::3]).max(), clamped_height)

def height_at(theta):
    return max_height(response(equilibrium, theta))

def loss(theta):
    return 0.5 * ((height_at(theta) - target_height) / height_scale).square()

initial_u = response(equilibrium, theta.detach())
initial_height = max_height(initial_u).item()
gradient, = torch.autograd.grad(height_at(theta), theta)
assert torch.isfinite(gradient).all() and gradient.norm() > 0
direction = gradient / gradient.norm()
h = 1e-5
central = (height_at(theta.detach() + h * direction)
           - height_at(theta.detach() - h * direction)) / (2 * h)
torch.testing.assert_close(gradient @ direction, central, rtol=2e-4, atol=2e-7)
print(f'Height derivative, adjoint / finite difference: {(gradient @ direction).item():.8g} / {central.item():.8g}')

## Check the fitted ear height

L-BFGS updates the full parameter vector under the same prescribed force.
Check the final height error and the force residual on the free DOFs.
A single height measurement permits many material parameter vectors, so
the fit demonstrates one way to reach the target height.

In [ ]:
optimizer = torch.optim.LBFGS(
    [theta], max_iter=20, tolerance_grad=1e-9, tolerance_change=1e-12,
    line_search_fn='strong_wolfe',
)

def closure():
    optimizer.zero_grad()
    value = loss(theta)
    value.backward()
    return value

optimizer.step(closure)
fitted_theta = theta.detach().numpy()
fitted_u = response(equilibrium, theta.detach())
fitted_height = max_height(fitted_u).item()
height_error = abs(fitted_height - target_height)
residual = np.linalg.norm(make_energy(fitted_theta).evaluate(fitted_u.numpy()).gradient()[~fixed_mask])
assert height_error < 1e-7
assert residual < 1e-9

print(f'Rest height:    {y.max():.9f}')
print(f'Initial height: {initial_height:.9f}')
print(f'Target height:  {target_height:.9f}')
print(f'Fitted height:  {fitted_height:.9f}')
print(f'Height error:   {height_error:.2e}')
print(f'Free equilibrium residual norm: {residual:.2e}')

## Compare the loaded bunny before and after fitting

Both overviews show the deformed surface in the same x–y view, with
matching axis limits and actual displacement scale. The close-up overlays
the loaded surface vertices for the two equilibria. Stars mark the highest
vertex of the whole bunny in each state. The dashed line marks the target.

In [ ]:
def deformed_positions(u):
    positions = bunny.positions.copy()
    positions[vertices] += u.numpy().reshape(-1, 3)
    return positions

before = deformed_positions(initial_u)
after = deformed_positions(fitted_u)
states = [
    ('Before optimization', before, '#285b91', '#d5e2f0'),
    ('After optimization', after, '#087d68', '#c8e7df'),
]
surface_faces = extracted_surface.mesh.topology.cells
loaded_vertices = vertices[loaded]
extent = np.vstack((before[:, :2], after[:, :2]))
lo, hi = extent.min(axis=0) - .005, extent.max(axis=0) + .005

fig, (before_ax, after_ax, detail) = plt.subplots(
    1, 3, figsize=(12.6, 4.8), width_ratios=(1, 1, 1.2), layout='constrained',
)
fig.suptitle('Bunny under the same upward ear load', fontweight='bold')
fig.get_layout_engine().set(h_pad=.10)

for ax, (label, positions, color, fill) in zip((before_ax, after_ax), states):
    xy = positions[:, :2]
    loaded_xy = xy[loaded_vertices]
    peak = xy[xy[:, 1].argmax()]
    ax.add_collection(PolyCollection(
        xy[surface][surface_faces], facecolors=fill, edgecolors='none', antialiaseds=False,
    ))
    ax.scatter(*loaded_xy.T, s=8, color=color, zorder=3)
    ax.axhline(target_height, color='0.4', linestyle='--', linewidth=1)
    ax.scatter(*peak, s=100, marker='*', color=color, edgecolors='white', zorder=4)
    ax.set(title=f'{label}\nHighest point {peak[1]:.6f}', xlabel='x',
           xlim=(lo[0], hi[0]), ylim=(lo[1], hi[1]), aspect='equal')
    ax.locator_params(axis='x', nbins=5)

    detail.scatter(*loaded_xy.T, s=28, color=color, alpha=.8, label=label, zorder=3)
    detail.scatter(*peak, s=145, marker='*', color=color, edgecolors='white', zorder=4)

before_ax.set_ylabel('Height y')
after_ax.tick_params(labelleft=False)
detail.axhline(target_height, color='0.4', linestyle='--', linewidth=1,
               label=f'Target height {target_height:.3f}')
detail.set(title='Ear-tip close-up', xlabel='x', ylabel='Height y', aspect='equal')
detail.locator_params(axis='x', nbins=5)
detail.grid(alpha=.2)
fig.legend(*detail.get_legend_handles_labels(), loc='outside lower center', ncols=3, frameon=False)
plt.show()

## Try it

**If materials reach the same height under an upward pull, how much can
their sideways displacement differ?**

1. Keep the full parameter fit above. Starting from lesson 5's weights,
   fit a single positive stiffness multiplier $s$ to reach $0.200$ under
   the upward load.
2. Make a second version of the lesson 5 material that resists volume
   change more strongly. Fit its own stiffness multiplier to the same
   height.
3. Rotate the force from $+y$ to $+x$. Keep its magnitude, the loaded
   vertices, and the clamp the same. Freeze all three fits and solve
   from rest. Compare the $x$ displacement and the perpendicular $z$
   drift of the vertex that was highest in the rest mesh.

### Calibrate two different volume responses to the same height

The final network layer is linear and has no bias. Multiplying its 32
weights by $s$ multiplies the network value and both reference correction
terms by $s$, giving $W_s=sW_{05}$.

For the stronger volume response, multiply the first layer's column
for $\log J$ by 4. This feeds $4\log J$ into the network while keeping
the two shape inputs the same. For a pure volume change, the quadratic
curvature at rest is then 16 times larger before the overall stiffness
multiplier is fitted. The reference correction still gives zero stress
at rest.

For each material, choose a bracket whose endpoint heights straddle
$0.200$. A scalar root finder adjusts the multiplier until the height
error is zero.

In [ ]:
def fit_stiffness(reference, bracket):
    def parameters(scale):
        scaled = reference.copy()
        scaled[-32:] *= scale
        return scaled

    def height_residual(scale):
        return height_at(torch.from_numpy(parameters(scale))).item() - target_height

    scale = brentq(height_residual, *bracket)
    height = target_height + height_residual(scale)
    assert abs(height - target_height) < 1e-7
    return parameters(scale), scale, height

scaled_theta, stiffness_scale, scaled_height = fit_stiffness(trained_theta, (.6, 1.))

volume_seed = trained_theta.copy()
first_layer = volume_seed[:32 * 3].reshape(32, 3)
first_layer[:, 2] *= 4.
volume_theta, volume_scale, volume_height = fit_stiffness(volume_seed, (.38, .42))
print(f'Fitted stiffness multiplier {stiffness_scale:.6f}')
print(f'Fitted multiplier with stronger volume response {volume_scale:.6f}')

### Pull sideways and measure the same vertex

Move the nodal force components from $y$ to $x$ and replace the external
potential. Start each solve at zero displacement with the same rest mesh
and clamp. Use the index of the highest rest vertex for every measurement.
The $x$ component measures how far it moves along the pull, and the $z$
component measures how far it drifts across the pull.

In [ ]:
lateral_load = np.zeros_like(load)
lateral_load[:, 0] = load[:, 1]
lateral_external = energy.QuadraticEnergy(zero_quadratic, -lateral_load.ravel())

tip = X[:, 1].argmax()
assert loaded[tip] and X[tip, 1] == y.max()
rest_tip_xz = X[tip, [0, 2]]
lateral_tip_positions = {}
print(f'Measured vertex {vertices[tip]} (original bunny index)')
print(f'{"Material fit":<26} {"Upward peak height":>19} {"Sideways x":>13} {"Cross-pull z":>13}')

for label, parameters, vertical_height in [
    ('All 1,216 parameters', fitted_theta, fitted_height),
    ('Stiffness multiplier', scaled_theta, scaled_height),
    ('Stronger volume response', volume_theta, volume_height),
]:
    potential = make_energy(parameters, lateral_external)
    result = solver.newton_solve(potential, fixed, zero_displacement, newton_options)
    assert result.status == solver.NewtonStatus.Converged
    residual = potential.evaluate(result.x).gradient()[~fixed_mask]
    assert np.linalg.norm(residual) < 1e-9
    tip_dx, _, tip_dz = result.x.reshape(-1, 3)[tip]
    lateral_tip_positions[label] = rest_tip_xz + [tip_dx, tip_dz]
    print(f'{label:<26} {vertical_height:>19.9f} {tip_dx:>13.9f} {tip_dz:>13.9f}')

### Compare the ear-tip positions in the horizontal plane

Each line connects the rest position to one fitted material's position
under the same sideways load. The inset magnifies the same $x$–$z$
coordinates around the three endpoints. Both views preserve the aspect ratio.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.6), layout='constrained')
zoom = ax.inset_axes([.08, .43, .48, .51])
ax.scatter(*rest_tip_xz, s=78, marker='x', color='0.2', linewidths=2,
           label='Rest position', zorder=4)

colors = ('#285b91', '#087d68', '#ad4f23')
markers = ('o', 's', '^')
for (label, position), color, marker in zip(lateral_tip_positions.items(), colors, markers):
    segment = np.vstack((rest_tip_xz, position))
    ax.plot(*segment.T, color=color, linewidth=1.5, alpha=.65)
    ax.scatter(*position, s=68, marker=marker, color=color, edgecolors='white',
               label=label, zorder=4)
    zoom.scatter(*position, s=80, marker=marker, color=color, edgecolors='white', zorder=4)

endpoints = np.array(list(lateral_tip_positions.values()))
positions = np.vstack((rest_tip_xz, endpoints))
x_span = np.ptp(positions[:, 0])
ax.set(xlim=(positions[:, 0].min() - .10 * x_span, positions[:, 0].max() + .12 * x_span),
       ylim=(positions[:, 1].min() - .14 * x_span, positions[:, 1].max() + .48 * x_span),
       xlabel='x (mesh units)', ylabel='z (mesh units)', aspect='equal',
       title=f'Ear-tip vertex {vertices[tip]} under the same sideways load')
ax.grid(alpha=.2)
ax.legend(loc='upper right', framealpha=1)

padding = max(.23 * np.ptp(endpoints, axis=0).max(), 1e-5)
lo, hi = endpoints.min(axis=0) - padding, endpoints.max(axis=0) + padding
zoom.set(xlim=(lo[0], hi[0]), ylim=(lo[1], hi[1]), aspect='equal',
         title='Deformed positions (zoom)')
zoom.title.set_fontsize(9)
zoom.tick_params(labelsize=8)
zoom.ticklabel_format(style='plain', useOffset=False)
zoom.locator_params(nbins=4)
zoom.grid(alpha=.2)
ax.indicate_inset_zoom(zoom, edgecolor='0.4', alpha=.55)
plt.show()